In [6]:
!pip uninstall -y langchain langchain-core langchain-community langchain-pinecone langchain-huggingface numpy


Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


In [2]:
!pip install pinecone pandas sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 14.5 MB/s eta 0:00:00


In [3]:
!pip install transformers accelerate openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=8b17f8299815f04d494db1a8a4b1e398bf0b5be3325dbaa04e96f2025b640bab
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [4]:
import os
import re
import torch
from typing import List, Dict, Any

from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import whisper

In [5]:
# Pinecone
PINECONE_API_KEY = "my api key"
PINECONE_INDEX_NAME = "assunnah-index"

# Models
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"
EMBEDDING_DIM = 1024
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # use 1.5B if needed
WHISPER_MODEL_NAME = "base"

# Retrieval
TOP_K = 3
SIMILARITY_THRESHOLD = 0.40
MIN_RELEVANT_DOCS = 1

In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding model loaded")

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
print("Connected to Pinecone")

tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
print("Qwen loaded")

whisper_model = whisper.load_model(WHISPER_MODEL_NAME)
print("Whisper loaded")

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 148MiB/s]


Whisper loaded


In [9]:
def embed_query(query: str) -> List[float]:
    query_instruction = f"Represent this sentence for searching relevant passages: {query}"
    embedding = embedding_model.encode(
        [query_instruction],
        normalize_embeddings=True
    )[0]
    return embedding.tolist()

In [10]:
def retrieve_chunks(query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    query_vector = embed_query(query)
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    return results.get("matches", [])

In [11]:
def format_context(matches: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, match in enumerate(matches, start=1):
        meta = match["metadata"]
        source = meta.get("source", "unknown")
        chunk_id = meta.get("chunk_id", "unknown")
        text = meta.get("text", "")
        blocks.append(f"[Source {i}] File: {source} | Chunk: {chunk_id}\n{text}")
    return "\n\n".join(blocks)

In [12]:
def build_rag_prompt(question: str, context: str) -> str:
    return f"""
You are a grounded question-answering assistant for the As-Sunnah Foundation knowledge base.

Answer the user's question using ONLY the retrieved context below.

Rules:
1. Use only the retrieved context.
2. Do not use outside knowledge.
3. If the answer is not clearly supported by the context, reply exactly:
I don’t know based on provided data
4. Keep the answer concise and factual.
5. Include important names, dates, numbers, and project details exactly if available.
6. End with source references in this format:
Sources: [Source 1], [Source 2]

Retrieved Context:
{context}

User Question:
{question}

Answer:
""".strip()

In [13]:
def generate_text(prompt: str, max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(qwen_model.device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    new_tokens = outputs[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [14]:
def rag_answer(question: str) -> Dict[str, Any]:
    matches = retrieve_chunks(question, top_k=TOP_K)

    if not matches:
        return {
            "question": question,
            "answer": "I don’t know based on provided data",
            "sources": []
        }

    top_score = float(matches[0].get("score", 0.0))
    if top_score < SIMILARITY_THRESHOLD:
        return {
            "question": question,
            "answer": "I don’t know based on provided data",
            "sources": []
        }

    context = format_context(matches)
    prompt = build_rag_prompt(question, context)
    answer = generate_text(prompt, max_new_tokens=256)

    if "i don’t know based on provided data" in answer.lower() or "i don't know based on provided data" in answer.lower():
        final_answer = "I don’t know based on provided data"
    else:
        final_answer = answer

    sources = []
    for match in matches:
        meta = match["metadata"]
        sources.append({
            "source": meta.get("source", "unknown"),
            "chunk_id": meta.get("chunk_id", "unknown"),
            "score": float(match.get("score", 0.0))
        })

    return {
        "question": question,
        "answer": final_answer,
        "sources": sources
    }

In [15]:
def transcribe_audio(audio_path: str) -> str:
    result = whisper_model.transcribe(audio_path)
    return result["text"].strip()

In [16]:
def ask_from_audio(audio_path: str) -> Dict[str, Any]:
    question = transcribe_audio(audio_path)
    result = rag_answer(question)
    result["transcribed_question"] = question
    return result

In [17]:
questions = [
    "who is ahmadullah?"
]

for q in questions:
    print("=" * 100)
    res = rag_answer(q)
    print("Question:", res["question"])
    print("Answer:", res["answer"])
    print("Sources:", res["sources"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: who is ahmadullah?
Answer: Ahmadullah is the chairman of As-Sunnah Foundation. He founded the organization in 2017 and is dedicated to education, da'wah, and overall humanitarian welfare.
Sources: [{'source': 'as_sunnah_foundation_extract.txt', 'chunk_id': 19, 'score': 0.52720356}, {'source': 'as_sunnah_foundation_extract.txt', 'chunk_id': 4, 'score': 0.498077393}, {'source': 'as_sunnah_foundation_extract.txt', 'chunk_id': 18, 'score': 0.438397408}]


In [16]:
from google.colab import files

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

result = ask_from_audio(audio_path)

print("Transcribed Question:", result["transcribed_question"])
print("Answer:", result["answer"])
print("Sources:", result["sources"])

Saving Recording (6).m4a to Recording (6).m4a
Transcribed Question: Can you please explain about if that distribution in a Suna Foundation?
Answer: The As-Sunnah Foundation distributes nutritious iftar during Ramadan to poor fasting individuals. This activity takes place in poverty-stricken areas across all districts of Bangladesh. In 2024, they provided 12, 030 packages of iftar to beneficiaries.
Sources: [{'source': 'as_sunnah_foundation_extract.txt', 'chunk_id': 21, 'score': 0.643353403}, {'source': 'as_sunnah_foundation_extract.txt', 'chunk_id': 13, 'score': 0.618423402}, {'source': 'as_sunnah_foundation_extract.txt', 'chunk_id': 0, 'score': 0.598422}]
